In [14]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

from structural_analysis.Material import Material
from structural_analysis.iterative_planform_sizing import size_planform

In [15]:
assumptions = Assumptions()

#TODO load the fuselage here
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

print(fixed.x_cg_max, fixed.x_cg_min, fixed.x_LE_wing, fixed.z_cg, fixed.z_tail_cone)
fixed.x_LE_wing = 1.255
delta_z = 0.01
fixed.z_tail_cone += delta_z
fixed.z_cg += delta_z
print(fixed.z_cg, fixed.z_tail_cone)
fixed.fuel_mass = 13.54
fixed.x_cg_min = 1.336 #m
fixed.x_cg_max = 1.403 #m
fixed.mass = 42.2 #kg 

for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

1.3589152301106575 1.271545179962353 1.4400000000000002 0.17957635338517247 0.11000000000000004
0.18957635338517248 0.12000000000000004


# Defining the standard aircraft with the standard planform
To be used when ppl don't wanna build their own planform

In [16]:
standard_wing = Planform(aspect_ratio=27, span=2.667, sweep_quarter_deg=15., taper=.5, thickness_to_chord=0.12, cm_quarter_chord=0,
                         wetted_surface_ratio=1.07, interference_factor=1.0, clmax=1.25, flap=False)

In [17]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

standard_wing.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
standard_wing.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
standard_wing.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
standard_wing.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [18]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

fuselage_diameter = fixed.fuselage.diameter_max
size_planform(planform=standard_wing, thicknesses=assumptions.allowable_thicknesses, fuselage_diameter=fuselage_diameter, material_skin=material_skin, density_core=assumptions.foam_denisty)
    

Stresses 42860371.45245816, 605594930.7107136, 0.0004
Stresses 28624990.457300115, 415981846.4171284, 0.0005959183673469389
Stresses 21433921.70716872, 320415537.54634386, 0.0007918367346938775
Stresses 17095508.328990243, 262939334.64787665, 0.0009877551020408164
Stresses 14193259.379450185, 224643733.746665, 0.0011836734693877551
Stresses 12115317.823862324, 197360696.28089306, 0.0013795918367346938


In [19]:
print(standard_wing.mass_cache, standard_wing.x_cg_cache, fixed.fuel_mass)

0.25583403427157053 0.21700554646188894 13.54


# We consider the thing to be tailed

In [20]:
tail = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=7., taper_h=.7, taper_v=.8).find_planforms(standard_wing)

print(tail[0].wing_area / standard_wing.wing_area)

Stresses 26973162.73787462, 126305995.3038049, 0.0004
Stresses 18077007.05226308, 89002659.99653415, 0.0005959183673469389
Stresses 100134303.22770756, 2316834881.303295, 0.0004
Stresses 67183548.33650962, 1894025696.4247384, 0.0005959183673469389
Stresses 50538321.63889414, 1738229715.865523, 0.0007918367346938775
Stresses 40496160.07752287, 1690091500.3366485, 0.0009877551020408164
Stresses 33778300.274398625, 1683778626.6799808, 0.0011836734693877551
Stresses 28968471.657960564, 1673943836.0906134, 0.0013795918367346938
Stresses 25354869.8476832, 1622699758.4021964, 0.0015755102040816327
Stresses 22540590.096822098, 1507690205.6488917, 0.0017714285714285716
Stresses 20286830.79426524, 1333606270.5150301, 0.0019673469387755105
Stresses 18441299.59179415, 1128565958.4144745, 0.002163265306122449
Stresses 16902292.602882296, 925324773.6480657, 0.002359183673469388
Stresses 15599299.465305187, 745660519.230673, 0.002555102040816327
Stresses 14481895.854741989, 597846320.5900711, 0.00275

In [21]:
for t in tail:
    t.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    t.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    t.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    t.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [22]:
ac = Aircraft(fixed, [standard_wing] + tail)


In [23]:
for acp in ac.planforms:
    print(acp.mass_cache)

print(ac.fixed.x_cg_min, ac.fixed.x_cg_max, ac.fixed.x_LE_wing)

0.25583403427157053
0.0029115349494355045
0.005713582343658339
1.336 1.403 1.255


# Requirement check for the aircraft

In [24]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [25]:
failed_reqs = list()
for requirement, label in zip(requirements, requirement_labels):
    if not requirement.assess(ac):
        failed_reqs.append(label)

if len(failed_reqs):
    print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
    print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
    print(f"Failed: {failed_reqs}")
    print()

In [26]:
#TODO: ctrl surface sizing